## Dataset Information
This dataset contains physicochemical properties of red and white wines, along with their quality ratings.

### Dataset Overview:
- **Source**: Vinho Verde wines from Portugal
- **Types**: Red and White wines
- **Quality Ratings**: Scale from 0 (worst) to 10 (best)
- **Total Features**: 12 input features + quality target
- **Purpose**: Predict wine quality based on physicochemical properties

### Features Description:
1. **fixed acidity**: Most acids involved with wine that do not evaporate readily
2. **volatile acidity**: Amount of acetic acid in wine (high amounts lead to vinegar taste)
3. **citric acid**: Found in small quantities, adds freshness and flavor
4. **residual sugar**: Amount of sugar remaining after fermentation
5. **chlorides**: Amount of salt in the wine
6. **free sulfur dioxide**: Prevents microbial growth and wine oxidation
7. **total sulfur dioxide**: Amount of free and bound forms of SO2
8. **density**: The density of wine is close to that of water
9. **pH**: Describes how acidic or basic a wine is
10. **sulphates**: A wine additive that contributes to SO2 levels
11. **alcohol**: Percentage of alcohol in wine
12. **type**: Red or White wine

### Target Variable:
- **quality**: Score between 0 and 10

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

: 

In [ ]:
# Load and display initial data information
df = pd.read_csv('winequality.csv')

In [ ]:
print("Dataset Shape:", df.shape)
print("\nFeature Information:")
display(df.info())

In [ ]:
print("\nSample Data:")
display(df.head())


In [ ]:
print("\nDetailed Statistics:")
display(df.describe())


In [ ]:
# Check class distribution
print("\nWine Type Distribution:")
display(df['type'].value_counts())

In [ ]:
print("\nQuality Distribution:")
display(df['quality'].value_counts().sort_index())

## Initial Data Analysis Insights

### Data Quality Check:
- No missing values in the dataset
- All features are numerical except 'type'
- Quality ratings are discrete values
- Dataset is imbalanced between red and white wines

### Statistical Observations:
1. **Quality Distribution**: Most wines are rated between 5-7
2. **Wine Types**: More white wines than red wines
3. **Feature Ranges**:
   - Alcohol content varies from ~8% to ~14%
   - pH ranges from ~2.7 to ~4.0
   - Residual sugar shows high variability


In [ ]:
# Enhanced Exploratory Data Analysis

# 1. Quality Distribution by Wine Type
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='quality', y='type', orient='h')
plt.title('Quality Distribution by Wine Type')
plt.show()

### 1. Quality Distribution by Wine Type
- White wines show slightly higher average quality
- Red wines have more consistent quality ratings
- Both types show normal-like distribution

In [ ]:
# 2. Feature Correlations with Quality
plt.figure(figsize=(15, 6))
numeric_features = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_features].corr()['quality'].sort_values(ascending=False)
sns.barplot(x=correlations.index, y=correlations.values)
plt.xticks(rotation=45)
plt.title('Feature Correlations with Quality')
plt.show()

### 2. Feature Correlations
- Alcohol shows strongest positive correlation with quality
- Volatile acidity shows negative correlation
- Density shows negative correlation
- pH has weak correlation with quality

In [ ]:
# 3. Alcohol vs Quality Analysis
plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x='quality', y='alcohol', hue='type')
plt.title('Alcohol Content vs Quality by Wine Type')
plt.show()

### 3. Alcohol vs Quality
- Higher quality wines tend to have higher alcohol content
- The relationship is more pronounced in white wines
- More variation in alcohol content for white wines

In [ ]:
# 4. Acidity Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.scatterplot(data=df, x='fixed acidity', y='pH', hue='type', ax=axes[0])
axes[0].set_title('Fixed Acidity vs pH')

sns.scatterplot(data=df, x='volatile acidity', y='quality', hue='type', ax=axes[1])
axes[1].set_title('Volatile Acidity vs Quality')

sns.scatterplot(data=df, x='citric acid', y='quality', hue='type', ax=axes[2])
axes[2].set_title('Citric Acid vs Quality')

plt.tight_layout()
plt.show()

### 4. Acidity Analysis
- Fixed acidity and pH show strong negative correlation
- Higher volatile acidity generally leads to lower quality
- Citric acid shows different patterns for red and white wines

In [ ]:
# 5. Sugar and Alcohol Relationship
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='alcohol', y='residual sugar', hue='type', size='quality')
plt.title('Residual Sugar vs Alcohol Content')
plt.show()

### 5. Sugar and Alcohol
- White wines show more variation in residual sugar
- Higher quality wines tend to have moderate sugar levels
- Red wines cluster at lower sugar levels

In [ ]:
# 6. Density Analysis
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df, x='density', hue='type', fill=True)
plt.title('Density Distribution by Wine Type')
plt.show()

### 6. Density Distribution
- Red wines generally have higher density
- White wines show more variation in density
- Density could be a good discriminator between wine types

In [ ]:
# Advanced Feature Engineering and Model Preparation

# Create interaction features
df['acidity_ratio'] = df['fixed acidity'] / df['volatile acidity']
df['sugar_to_alcohol'] = df['residual sugar'] / df['alcohol']
df['so2_ratio'] = df['free sulfur dioxide'] / df['total sulfur dioxide']

In [ ]:
# Create polynomial features for key variables
df['alcohol_squared'] = df['alcohol'] ** 2
df['pH_squared'] = df['pH'] ** 2

In [ ]:
# Encode wine type
df['type_encoded'] = (df['type'] == 'red').astype(int)

In [ ]:
# Prepare features for modeling
features = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
           'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
           'pH', 'sulphates', 'alcohol', 'type_encoded', 'acidity_ratio',
           'sugar_to_alcohol', 'so2_ratio', 'alcohol_squared', 'pH_squared']

X = df[features]
y = df['quality']

In [ ]:
# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Model Building and Evaluation

def evaluate_and_visualize_model(model, X_train, X_test, y_train, y_test, model_name):
    # Train and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    # Print metrics
    print(f"\n{model_name} Performance Metrics:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R² Score: {r2:.4f}")
    print(f"Cross-validation scores: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

    # Visualizations
    plt.figure(figsize=(15, 5))

    # Actual vs Predicted
    plt.subplot(1, 2, 1)
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Actual Quality')
    plt.ylabel('Predicted Quality')
    plt.title(f'{model_name}: Actual vs Predicted')

    # Prediction Error Distribution
    plt.subplot(1, 2, 2)
    errors = y_test - y_pred
    sns.histplot(errors, kde=True)
    plt.xlabel('Prediction Error')
    plt.ylabel('Count')
    plt.title('Error Distribution')

    plt.tight_layout()
    plt.show()

    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'CV Score': cv_scores.mean()}

In [ ]:
# Train and evaluate models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    result = evaluate_and_visualize_model(model, X_train_scaled, X_test_scaled,
                                        y_train, y_test, name)
    results.append(result)


In [ ]:
# Compare models
results_df = pd.DataFrame(results)
display(results_df)

## Model Performance Analysis

### Model Comparison:
1. **Linear Regression**
   - Provides baseline performance
   - Shows linear relationships in data
   - Limited by assumption of linearity

2. **Decision Tree**
   - Captures non-linear relationships
   - More prone to overfitting
   - Good for feature importance analysis

3. **Random Forest**
   - Best overall performance
   - More robust to outliers
   - Better generalization

### Error Analysis:
- Error distributions show model reliability
- Most errors within ±1 quality point
- Some systematic errors in extreme qualities

### Key Findings:
1. Random Forest performs best due to:
   - Ability to capture non-linear relationships
   - Ensemble learning reducing overfitting
   - Handling feature interactions well

2. Feature importance varies by model:
   - Alcohol content consistently important
   - Acidity measures significant
   - Created interaction features improve predictions

3. Model Limitations:
   - Difficulty predicting extreme qualities
   - Some bias towards mean values
   - Limited by subjective nature of ratings

In [ ]:
# Feature Importance Analysis for Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature')
plt.title('Top 10 Most Important Features (Random Forest)')
plt.show()

In [ ]:
print("\nTop 5 Most Important Features:")
display(feature_importance.head())

### Key Insights:
1. **Wine Quality Drivers**
   - Alcohol content is the strongest predictor
   - Acidity balance is crucial
   - Chemical properties have clear impact on quality

2. **Type Differences**
   - White and red wines show distinct patterns
   - Different quality drivers for each type
   - Processing should consider wine type

3. **Model Selection**
   - Random Forest is most reliable
   - Good balance of accuracy and interpretability
   - Robust across different wine types

### Recommendations:
1. **For Winemakers**
   - Focus on alcohol content optimization
   - Monitor acidity levels carefully
   - Consider type-specific quality factors

2. **For Model Improvement**
   - Collect more data on extreme qualities
   - Include more sensory features
   - Consider separate models for each wine type

3. **For Future Research**
   - Investigate temporal effects
   - Study regional variations
   - Include price-quality relationships